# 08. 特徴ベクトルの最終組み立て

`01`〜`07`で確定した処理（RIASEC補完、認知度ラベル、職業タグ、教科対応）を
`data/processed/jobs.csv` に一つの成果物としてまとめる。

質問クイズ用の変換パラメータ（`riasec_transform.json`）は、スワイプ形式・
学習記録形式への転換に伴い不要になったため、このノートブックでは生成しない
（生成していた過去の版はgit履歴を参照。設計判断の経緯はdocs/design.md）。


In [1]:
import json
import re

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from ipd_loader import load_description, load_numeric

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

desc, desc_labels = load_description()
num, num_labels = load_numeric()
names = num[num.columns[1]]


## RIASEC補完（02と同じ手順）

In [2]:
riasec_cols = [c for c in num.columns if re.match(r"IPD_04_01_", str(c))]
know_cols = [c for c in num.columns if re.match(r"IPD_04_04_01_", str(c))]
work_cols = [c for c in num.columns if re.match(r"IPD_04_05_", str(c))]

riasec = num[riasec_cols].apply(pd.to_numeric, errors="coerce")
know = num[know_cols].apply(pd.to_numeric, errors="coerce")
work = num[work_cols].apply(pd.to_numeric, errors="coerce")

riasec_observed = ~riasec.isna().all(axis=1)
no_input = know.isna().all(axis=1) & work.isna().all(axis=1)
predictable = (~riasec_observed) & ~no_input

def scale_domain(domain_df):
    scaler = StandardScaler()
    filled = domain_df.fillna(domain_df.mean())
    scaled = pd.DataFrame(scaler.fit_transform(filled), columns=domain_df.columns, index=domain_df.index)
    return scaled.fillna(0.0)

know_scaled = scale_domain(know)
work_scaled = scale_domain(work)
X_all = pd.concat([know_scaled, work_scaled], axis=1)
X_train = X_all[riasec_observed]
y_train = riasec[riasec_observed]
riasec_model = Ridge(alpha=1.0).fit(X_train.values, y_train.values)

riasec_full = riasec.copy()
X_pred = X_all[predictable]
riasec_full.loc[predictable, riasec_cols] = riasec_model.predict(X_pred.values)

unavailable = (~riasec_observed) & no_input
riasec_source = pd.Series("observed", index=num.index)
riasec_source[predictable] = "predicted"
riasec_source[unavailable] = "unavailable"
riasec_source.value_counts()


observed       482
predicted       29
unavailable      7
Name: count, dtype: int64

## 認知度ラベルと突き合わせ、167件の推薦対象を確定する（03/04と同じ手順）

In [3]:
awareness = pd.read_csv("../data/processed/awareness_scores.csv", index_col=0)
name_to_numidx = pd.Series(num.index, index=names)

recommendable_awareness = awareness[awareness["recommendable"]]
rows = []
for job_idx, row in recommendable_awareness.iterrows():
    hit = name_to_numidx.get(row["職業名"])
    if hit is None:
        continue
    if isinstance(hit, pd.Series):
        hit = hit.iloc[0]
    rows.append({"job_id": int(hit), "awareness_score": row["awareness_score"], "awareness_label": row["awareness_label"]})

job_pool = pd.DataFrame(rows).set_index("job_id")
job_pool = job_pool[riasec_source.loc[job_pool.index] != "unavailable"]
print("最終的な推薦対象:", len(job_pool))


最終的な推薦対象: 167


## RIASECの標準化・PCA（05と同じ、167件基準）

In [4]:
riasec_scaler = StandardScaler().fit(riasec.loc[riasec_observed, riasec_cols])

riasec_167 = riasec_full.loc[job_pool.index, riasec_cols]
riasec_167_z = pd.DataFrame(
    riasec_scaler.transform(riasec_167.values), columns=riasec_cols, index=riasec_167.index
)
riasec_167_z.describe()


/Users/oobasouma/yumetane/api/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,IPD_04_01_001,IPD_04_01_002,IPD_04_01_003,IPD_04_01_004,IPD_04_01_005,IPD_04_01_006
count,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000
mean,0.048055,0.064995,0.026642,-0.034886,-0.012321,-0.057923
std,0.970324,1.013911,1.016259,1.013139,0.940693,0.928341
min,-2.363509,-2.014206,-2.103988,-2.511912,-2.465584,-3.135349
25%,-0.572431,-0.735243,-0.652677,-0.734801,-0.617937,-0.655598
50%,0.022717,0.034013,-0.173546,-0.162788,0.028462,-0.003205
75%,0.689379,0.735168,0.460657,0.685055,0.529213,0.435558
max,2.308415,2.709687,3.381170,2.900845,2.622380,3.697524


## jobs.csv を組み立てる

In [5]:
desc_name_indexed = desc.set_index(desc[desc.columns[1]])

jobs = pd.DataFrame({
    "job_name": names.loc[job_pool.index],
    "riasec_source": riasec_source.loc[job_pool.index],
    "awareness_score": job_pool["awareness_score"],
    "awareness_label": job_pool["awareness_label"],
})

for c in riasec_cols:
    jobs[f"riasec_{num_labels[c]}"] = riasec_167.loc[jobs.index, c].values
    jobs[f"riasec_{num_labels[c]}_z"] = riasec_167_z.loc[jobs.index, c].values

# 職業タグ用に、仕事の性質39項目のうち選定した19項目のzスコアを追加する
# （選定基準・カテゴリ分けはdocs/vocab.md「職業タグ」参照）
TAG_WORK_CONTEXT_ITEMS = {
    "IPD_04_05_016": "外で働く",
    "IPD_04_05_017": "座って集中",
    "IPD_04_05_018": "立ち仕事",
    "IPD_04_05_031": "歩き回る仕事",
    "IPD_04_05_001": "人と関わる",
    "IPD_04_05_007": "チームで動く",
    "IPD_04_05_008": "お客さんと話す",
    "IPD_04_05_009": "みんなをまとめる",
    "IPD_04_05_006": "スピード勝負",
    "IPD_04_05_023_01": "決まった予定で動く",
    "IPD_04_05_023_02": "毎日ちがう",
    "IPD_04_05_013": "結果に責任を持つ",
    "IPD_04_05_035": "人の安全を守る",
    "IPD_04_05_010": "正確さが大事",
    "IPD_04_05_011": "コツコツ続ける",
    "IPD_04_05_021": "自分で決められる",
    "IPD_04_05_029": "特別な装備を使う",
    "IPD_04_05_032": "手先を使う",
    "IPD_04_05_034": "機械と働く",
}
assert len(TAG_WORK_CONTEXT_ITEMS) == 19

# 02/08で使ってきたscale_domainと同じ手順（NaNは列平均で埋めてから標準化）で、
# 518職業全体を母集団として標準化する
work_filled = work.fillna(work.mean())
work_scaler = StandardScaler().fit(work_filled)
work_z_full = pd.DataFrame(
    work_scaler.transform(work_filled), columns=work.columns, index=work.index,
)
for code, label in TAG_WORK_CONTEXT_ITEMS.items():
    jobs[f"workctx_{label}_z"] = work_z_full.loc[jobs.index, code].values

# 学習記録が見つける職業を選ぶための、知識33項目→教科スコア。
# 対応基準・却下した項目（経済学会計学、法律学政治学など）はdocs/design.md
# 「知識33項目 → 教科への対応」参照。1教科に複数項目がある場合はzスコアの平均。
SUBJECT_TO_KNOWLEDGE_ITEMS = {
    "数学": ["IPD_04_04_01_015"],
    "理科": ["IPD_04_04_01_016", "IPD_04_04_01_017", "IPD_04_04_01_018"],
    "社会": ["IPD_04_04_01_021", "IPD_04_04_01_028"],
    "国語": ["IPD_04_04_01_025"],
    "英語": ["IPD_04_04_01_026"],
    "美術・音楽": ["IPD_04_04_01_027"],
    "技術・家庭": [
        "IPD_04_04_01_008", "IPD_04_04_01_009", "IPD_04_04_01_010",
        "IPD_04_04_01_011", "IPD_04_04_01_012", "IPD_04_04_01_013",
        "IPD_04_04_01_014", "IPD_04_04_01_032",
    ],
}

know_filled = know.fillna(know.mean())
know_scaler = StandardScaler().fit(know_filled)
know_z_full = pd.DataFrame(
    know_scaler.transform(know_filled), columns=know.columns, index=know.index,
)
for subject, codes in SUBJECT_TO_KNOWLEDGE_ITEMS.items():
    jobs[f"subject_{subject}_z"] = know_z_full.loc[jobs.index, codes].mean(axis=1).values

descriptions = []
for name in jobs["job_name"]:
    if name in desc_name_indexed.index:
        row = desc_name_indexed.loc[name]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        descriptions.append(row.get("IPD_03_01_000", ""))
    else:
        descriptions.append("")
jobs["description"] = descriptions

jobs = jobs.reset_index().rename(columns={"index": "job_id"})
jobs.to_csv("../data/processed/jobs.csv", index=False, encoding="utf-8-sig")
print("保存件数:", len(jobs), " 列数:", len(jobs.columns))
jobs.head()


保存件数: 167  列数: 44


,job_id,job_name,riasec_source,awareness_score,awareness_label,riasec_現実的,riasec_現実的_z,riasec_研究的,riasec_研究的_z,riasec_芸術的,...,workctx_手先を使う_z,workctx_機械と働く_z,subject_数学_z,subject_理科_z,subject_社会_z,subject_国語_z,subject_英語_z,subject_美術・音楽_z,subject_技術・家庭_z,description
0,2,洋菓子製造、パティシエ,observed,2,知っている,3.345,0.022717,3.000,-0.339886,2.945,...,1.229071e+00,0.084322,-0.164561,-0.295868,-0.813167,-0.972902,-0.780988,-0.127304,-0.108852,洋菓子店や菓子工場で洋菓子を製造する。
1,11,ハム・ソーセージ・ベーコン製造,observed,0,知らない,3.304,-0.085748,2.739,-0.931222,2.543,...,8.435530e-01,-0.487023,-0.871708,-0.499513,-0.970207,-1.064121,-1.066927,-0.729336,-0.148921,原料肉を分割・整形した後、塩づけ、くん煙などの加工をして、ハム・ソーセージ・ベーコンを製造する。
2,12,ワイン製造,observed,1,名前は聞いたことがある,3.708,0.983028,3.292,0.321686,2.958,...,6.028316e-16,0.000000,1.392683,1.946680,2.155544,0.244190,1.074779,0.902299,1.372984,ブドウからワインを醸造する作業に従事する。
3,13,ビール製造,observed,1,名前は聞いたことがある,3.594,0.681443,3.469,0.722707,3.031,...,-1.351728e-01,1.406089,0.217146,0.174339,-0.679642,-0.456825,-0.381807,-0.679286,0.417060,ビール醸造所（ブルワリー）において、ビールの製造に従事する。
4,15,野菜つけ物製造,observed,0,知らない,3.382,0.120600,2.818,-0.752235,2.545,...,6.019258e-01,-0.805384,-0.496084,-0.148103,-0.303472,-1.145344,-0.786651,-0.549155,0.263999,野菜を材料にしたつけ物を製造するため、材料の選別、洗浄、カット、漬け込み、塩抜き、計量、殺菌、検査、包装などの作...
